# **Bibliotecas**

In [2]:
# Instalação das dependências do Pipeline de Polarimetria
%pip install "numpy<2" scipy pandas matplotlib astropy photutils astropop

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
types-geopandas 1.1.2 requires pandas-stubs, which is not installed.
types-geopandas 1.1.2 requires pyproj, which is not installed.
types-seaborn 0.13.2 requires pandas-stubs, which is not installed.
types-pycocotools 2.0 requires numpy>=2.0.0rc1, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install --force-reinstall --no-deps matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.8 MB)
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.9
    Uninstalling matplotlib-3.10.9:
      Successfully uninstalled matplotlib-3.10.9
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --force-reinstall matplotlib kiwisolver pillow cycler pyparsing contourpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached kiwisolver-1.5.1-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached fonttools-4.65.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (125 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)


In [5]:
%pip install --force-reinstall pillow

Defaulting to user installation because normal site-packages is not writeable
  Using cached pillow-12.3.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 12.3.0
    Uninstalling pillow-12.3.0:
      Successfully uninstalled pillow-12.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
types-seaborn 0.13.2 requires pandas-stubs, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import glob
import warnings
import concurrent.futures
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd
import shutil

In [7]:
from astropy.io import fits
from astropy import units as u
from astropy.utils.exceptions import AstropyWarning
from astropy.stats import sigma_clipped_stats, sigma_clip

In [8]:
from astropop.image import imcombine, processing, imarith
from astropop.framedata import read_framedata
from astropop.file_collection import FitsFileGroup
from astropop.polarimetry import SLSDualBeamPolarimetry

In [9]:
from photutils.detection import DAOStarFinder
from photutils.centroids import centroid_com
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry

/home/likeableng/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from collections import defaultdict
from matplotlib.patches import Circle

In [11]:
# Silencia os avisos do HIERARCH
warnings.simplefilter('ignore', category=AstropyWarning)

# **Caminho dos Dados**

In [12]:
#Caminho até chegar chegar antes na pasta da noite
CAMINHO_PASTA__REDUCAO = '/home/likeableng/BeACoN/arquivos_de_reducao/'

In [13]:
#Colocar os dados da noite
DATA = "20250719" #ex: "AAAAMMDD"
NOME_ESTRELA = "48lib" #ex: "paqr" "osco"

# **Ajuste Pré redução**

In [14]:
#Ajuste data
mes = int(DATA[4:6])

nome_mes = ['', 'jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 
            'ago', 'set', 'out', 'nov', 'dez']

data_ajustada = DATA[2:4]+nome_mes[mes]+DATA[6:]

In [15]:
#Caminhos para pastas reduced e raw

raw_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/raw/"

reduced_dir = f"{CAMINHO_PASTA__REDUCAO}{data_ajustada}/reduced/"
os.makedirs(reduced_dir, exist_ok=True)

In [16]:
#Caminho para dados estrela
pasta_raw_estrela = f'{raw_dir}{NOME_ESTRELA}'
pasta_reduced_estrela = f'{reduced_dir}{NOME_ESTRELA}'
os.makedirs(pasta_reduced_estrela, exist_ok=True)

#Caminho para dados calib
pasta_raw_calib = f'{raw_dir}calib/'
pasta_reduced_calib = f'{reduced_dir}calib/'
os.makedirs(pasta_reduced_calib, exist_ok=True)

# **Funções Motoras**

In [17]:
#Ajustar nome dos arquivos, alterando o filtro de F# para b,v,r,i

mapeamento = {
    'F1': 'b',
    'F2': 'v',
    'F3': 'r',
    'F4': 'i'
}

def ajustar_nome_filtro(caminho_da_pasta):
# Percorre todos os arquivos dentro da pasta especificada
    for nome_arquivo in os.listdir(caminho_da_pasta):
        if nome_arquivo.endswith('.fits'):
            for chave, letra in mapeamento.items():
                marcador_antigo = f'_{chave}_'
                marcador_novo = f'_{letra}_'
            
                if marcador_antigo in nome_arquivo:
                    # É preciso juntar o caminho da pasta com o nome do arquivo
                    caminho_antigo = os.path.join(caminho_da_pasta, nome_arquivo)
                    novo_nome = nome_arquivo.replace(marcador_antigo, marcador_novo)
                    caminho_novo = os.path.join(caminho_da_pasta, novo_nome)
                
                    os.rename(caminho_antigo, caminho_novo)
                    print(f'{nome_arquivo}  ->  {novo_nome}')
                    break


In [18]:
#Transformar imagens 3D em 2D

def descompactar_cubos_3d(pasta):    
    # Achar arquivos FITS na pasta de entrada
    arquivos_fits = glob.glob(os.path.join(pasta, '*.fits'))
    print(f"Encontrados {len(arquivos_fits)} arquivos na pasta de entrada.\n")
    
    for caminho_arquivo in arquivos_fits:
        nome_base = os.path.basename(caminho_arquivo).replace('.fits', '')

        try:
            # Ignora o ficheiro se estiver vazio (0 bytes) para evitar OSError
            if os.path.getsize(caminho_arquivo) == 0:
                print(f"Ignorando '{nome_base}.fits' (Ficheiro vazio/0 bytes).")
                continue

            dados, header = fits.getdata(caminho_arquivo, header=True)
            
            # Verifica a dimensão da matriz
            if dados.ndim == 3:
                num_imagens = dados.shape[0]
                print(f"Descompactando '{nome_base}.fits' ({num_imagens} fatias)...")
                
                for i in range(num_imagens):
                    fatia_2d = dados[i]
                    
                    # Cria um nome sequencial (ex: bias_001_001.fits)
                    nome_saida = f"{nome_base}_{i+1:03d}.fits"
                    caminho_saida = os.path.join(pasta, nome_saida)
                    
                    # Salva a fatia 2D com o cabeçalho original.
                    fits.writeto(caminho_saida, fatia_2d, header, overwrite=True)
                    
            elif dados.ndim == 2:
                print(f"Ignorando '{nome_base}.fits' (Já é 2D).")
            else:
                print(f"Ignorando '{nome_base}.fits' (Formato não suportado).")
                
        except OSError as e:
            # Captura especificamente ficheiros corrompidos ou não suportados pelo astropy
            print(f"Erro ao ler '{nome_base}.fits': Ficheiro corrompido ou inválido. A ignorar...")
        except Exception as e:
            # Previne que outros erros parem o loop
            print(f"Erro inesperado no ficheiro '{nome_base}.fits': {e}")
            
    print("\nDescompactação concluída!")


In [19]:
#Fazer o Master BIAS - sem usar a extração de raios cósmicos

def criar_master_bias(pasta_entrada, pasta_saida):
    print("\n[Calibração] Criando Master Bias por PREAMP (otimizado para velocidade)...")
    fg = FitsFileGroup(location=pasta_entrada, fits_ext=['.fits'], ext=0)
    
    bias_files = fg.filtered({'OBJECT': 'bias'}).files
    
    arquivos_por_preamp = defaultdict(list)
    for arq in bias_files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            # Pega o valor original (ex: '5.1x' ou '1x')
            preamp_raw = str(hdr.get('PREAMP', 'UNKNOWN'))
            
            # Limpa o texto: tira o 'x' e corta tudo depois do ponto
            preamp_val = preamp_raw.lower().replace('x', '').split('.')[0]
            
            arquivos_por_preamp[preamp_val].append(arq)
            
    for preamp_val, arq_2d in arquivos_por_preamp.items():
        print(f"-> Processando Bias para PREAMP: {preamp_val} ({len(arq_2d)} arquivos)")
        
        frames = [read_framedata(f, unit='adu', use_memmap_backend=True) for f in arq_2d]
        gain = float(frames[0].header['GAIN']) * u.electron/u.adu
        
        for frame in frames:
            processing.gain_correct(frame, gain, inplace=True)
            # Sem L.A. Cosmic para processamento rápido
            
        master = imcombine(frames, method='median')
        master.meta['astropop master_type'] = 'bias'
        master.meta['PREAMP'] = preamp_val
        
        sufixo_preamp = f"_g{preamp_val}" if preamp_val != '1' else ""
        caminho_out = os.path.join(pasta_saida, f'master_bias{sufixo_preamp}.fits')
        
        master.write(caminho_out, overwrite=True, no_fits_standard_units=True)
        print(f"   Salvo: {caminho_out}\n")


In [20]:
#Fazer o Mater Flat - sem usar a extração de raios cósmicos

letra_filtro = ["", "b", "v", "r", "i"]

def criar_master_flat(pasta_in, pasta_out):
    main_fg = FitsFileGroup(location=pasta_in, fits_ext=['.fits'], ext=0)
    flat_files = main_fg.filtered({'OBJECT': 'flat'}).files

    # 1. Agrupamento manual duplo (por Filtro e Preamp limpo)
    arquivos_por_grupo = defaultdict(list)
    for arq in flat_files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            # Extrai e limpa o PREAMP (ex: '5.1x' vira '5')
            preamp_raw = str(hdr.get('PREAMP', 'UNKNOWN'))
            preamp_val = preamp_raw.lower().replace('x', '').split('.')[0]
            
            # Extrai o filtro
            filtro_val = str(hdr.get('FILTER', 'UNKNOWN'))
            
            # Agrupa os dois juntos
            arquivos_por_grupo[(filtro_val, preamp_val)].append(arq)

    # 2. Processa cada grupo separadamente
    for (filtro_val, preamp_val), arquivos_flat_2d in arquivos_por_grupo.items():
        print(f"\n--- Processando Master Flat | Filtro: {filtro_val} | PREAMP: {preamp_val} ({len(arquivos_flat_2d)} arquivos) ---")
        
        # Define sufixo e carrega o Bias correto
        sufixo_preamp = f"_g{preamp_val}" if preamp_val != '1' else ""
        caminho_master_bias = os.path.join(pasta_out, f'master_bias{sufixo_preamp}.fits')
        
        if not os.path.exists(caminho_master_bias):
            print(f"ERRO: {caminho_master_bias} não encontrado! Abortando este flat.")
            continue
            
        master_bias = read_framedata(caminho_master_bias)

        flat_frames = [read_framedata(f, unit='adu', use_memmap_backend=True) for f in arquivos_flat_2d]
        gain = float(flat_frames[0].header['GAIN']) * u.electron / u.adu

        for f_idx, frame in enumerate(flat_frames):
            print(f" -> Processando flat [{filtro_val}] {f_idx+1}/{len(flat_frames)}", end='\r')
            processing.gain_correct(frame, gain, inplace=True)
            processing.subtract_bias(frame, master_bias, inplace=True)

        print(f"\nCombinando e normalizando Master Flat ({filtro_val}, PREAMP {preamp_val})...")
        master_flat = imcombine(flat_frames, method='median')
        mean_value = master_flat.mean()
        master_flat = imarith(master_flat, mean_value, '/')

        master_flat.meta['astropop master_type'] = 'flat'
        master_flat.meta['FILTER'] = filtro_val
        master_flat.meta['PREAMP'] = preamp_val

        valor_filtro = filtro_val[1:]
        numero_filtro = int(valor_filtro)
        letra = letra_filtro[numero_filtro]

        # Salva com o nome correto
        nome_out = f'master_flat{sufixo_preamp}_{letra}.fits'
        
        mflat_path = os.path.join(pasta_out, nome_out)
        master_flat.write(mflat_path, overwrite=True, no_fits_standard_units=True)

        print(f"Salvo: {mflat_path}")
        del flat_frames

In [21]:
#Função auxiliar da redução de imagens em paralelo
def dividir_em_lotes(lista, tamanho):
    return [lista[i:i + tamanho] for i in range(0, len(lista), tamanho)]

def processar_lote_por_filtro(filtro, preamp, arquivos, caminho_mbias, caminho_mflat, pasta_saida):
    master_bias = read_framedata(caminho_mbias)
    master_flat = read_framedata(caminho_mflat)
    
    total = len(arquivos)
    for i, arq in enumerate(arquivos):
        nome_arq = os.path.basename(arq)
        
        frame = read_framedata(arq, unit='adu', use_memmap_backend=True)
        gain = float(frame.header['GAIN']) * u.electron / u.adu

        processing.cosmics_lacosmic(frame, inplace=True)
        processing.gain_correct(frame, gain, inplace=True)
        processing.subtract_bias(frame, master_bias, inplace=True)
        processing.flat_correct(frame, master_flat, inplace=True)

        frame.meta['REDUCED'] = True
        
        # AQUI ADICIONAMOS O PREFIXO "red_"
        caminho_saida = os.path.join(pasta_saida, f"red_{nome_arq}")
        
        frame.write(caminho_saida, overwrite=True, no_fits_standard_units=True)
        
    return f"Lote [Filtro {filtro} | PREAMP {preamp}] ({total} imagens) CONCLUÍDO!"

In [22]:
#Realiza a calibração da Imagem com o MASTER BIAS e FLAT

def reduzir_objetos_paralelo(pasta_raw_obj, pasta_red_calib, pasta_red_obj):
    os.makedirs(pasta_red_obj, exist_ok=True)

    main_fg = FitsFileGroup(location=pasta_raw_obj, fits_ext=['.fits'], ext=0)
    
    # Agrupamento seguro sem utilizar o .group_by() do astropop
    arquivos_por_grupo = defaultdict(list)
    for arq in main_fg.files:
        hdr = fits.getheader(arq)
        if hdr.get('NAXIS') == 2:
            # Limpa o preamp (ex: '5.1x' vira '5')
            preamp_raw = str(hdr.get('PREAMP', 'UNKNOWN'))
            preamp_val = preamp_raw.lower().replace('x', '').split('.')[0]
            
            filtro_val = str(hdr.get('FILTER', 'UNKNOWN'))
            arquivos_por_grupo[(filtro_val, preamp_val)].append(arq)

    print("Iniciando processamento paralelo em pequenos lotes seguros...")

    with concurrent.futures.ProcessPoolExecutor() as executor:
        futuros = []
        
        for (filtro_val, preamp_val), arquivos_2d in arquivos_por_grupo.items():
            if not arquivos_2d:
                continue
            
            sufixo_preamp = f"_g{preamp_val}" if preamp_val != '1' else ""

            # Lógica para extrair a letra a partir da variável letra_filtro
            valor_filtro = filtro_val[1:]
            numero_filtro = int(valor_filtro)
            letra = letra_filtro[numero_filtro]
            
            caminho_mbias = os.path.join(pasta_red_calib, f'master_bias{sufixo_preamp}.fits')
            caminho_mflat = os.path.join(pasta_red_calib, f'master_flat{sufixo_preamp}_{letra}.fits')
            
            if not os.path.exists(caminho_mbias) or not os.path.exists(caminho_mflat):
                print(f"[AVISO] Calibração ausente para Filtro {filtro_val} ({letra}), PREAMP {preamp_val}. Pulando conjunto.")
                continue
            
            lotes = dividir_em_lotes(arquivos_2d, 50)
            
            for lote in lotes:
                futuro = executor.submit(processar_lote_por_filtro, filtro_val, preamp_val, lote, caminho_mbias, caminho_mflat, pasta_red_obj)
                futuros.append(futuro)

        for futuro in concurrent.futures.as_completed(futuros):
            print(futuro.result())

    print("\n\nRedução de TODOS os objetos concluída com sucesso!")


In [23]:
#Rastrear os centros para par
def rastrear_centroide_ancorado(dados, x_ancora, y_ancora, janela=12):
    """
    Rastreia a estrela SEM acumular erro. Procura sempre ao redor da âncora original.
    Subtrai o fundo local para não ser puxado por gradientes de ruído.
    """
    x_int, y_int = int(round(x_ancora)), int(round(y_ancora))
    h, w = dados.shape
    
    y_min, y_max = max(0, y_int - janela), min(h, y_int + janela + 1)
    x_min, x_max = max(0, x_int - janela), min(w, x_int + janela + 1)
    
    recorte = dados[y_min:y_max, x_min:x_max]
    recorte = np.nan_to_num(recorte, nan=np.nanmedian(recorte))
    
    # Subtrai mediana local para o centróide focar no pico da estrela
    recorte_sub = np.maximum(recorte - np.median(recorte), 0)
    xc_rel, yc_rel = centroid_com(recorte_sub)
    
    x_novo = x_min + xc_rel
    y_novo = y_min + yc_rel
    
    # Se o desvio for fora do limite aceitável, segura na posição de âncora
    if np.abs(x_novo - x_ancora) > (janela - 2) or np.abs(y_novo - y_ancora) > (janela - 2):
        return x_ancora, y_ancora
        
    return x_novo, y_novo

# **Calibração**

In [24]:
#Ajustar nome F# na pasta calib
ajustar_nome_filtro(pasta_raw_calib)

In [26]:
#Ajustar nome F# na pasta da estrela
ajustar_nome_filtro(pasta_raw_estrela)

FileNotFoundError: [Errno 2] No such file or directory: '/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/raw/48lib'

# **Ajuste de Quantidade de Observações do Objeto na Noite**

In [27]:
#Achar arquivos no filtro b, para verificar a quantidade de lâminas

padrao_busca = os.path.join(pasta_raw_estrela, f'{NOME_ESTRELA}_b_0*.fits')
arquivos_encontrados = sorted(glob.glob(padrao_busca))

In [28]:
#Verificando as lâminas
valores_lamina = []

for arq in arquivos_encontrados:
    try:
        hdr = fits.getheader(arq)

        # O valor -999.0 serve como aviso caso a palavra-chave não exista
        valor = hdr.get('LAMINA', -999.0)

        valor = valor[1:]

        if valor != -999.0:
            valores_lamina.append(valor)
            
    except Exception as e:
        print(f"Erro ao ler o ficheiro {os.path.basename(arq)}: {e}")

if valores_lamina:
    valor_mais_alto = max(valores_lamina)
    vezes_repetido = valores_lamina.count(valor_mais_alto)
    
    print("=== RESULTADOS DA LÂMINA ===")
    print(f"Total de leituras bem-sucedidas: {len(valores_lamina)}")
    print(f"Valor mais alto encontrado: {valor_mais_alto}")
    print(f"O valor mais alto aparece {vezes_repetido} vez(es) nas imagens.")

else:
    pastas = [
        f
        for f in os.listdir(pasta_raw_estrela)
        if os.path.isdir(os.path.join(pasta_raw_estrela, f))
    ]
    vezes_repetido = len(pastas)
    #print(vezes_repetido)

    padrao_busca = os.path.join(f"{pasta_raw_estrela}//Obs_1_{NOME_ESTRELA}", f'{NOME_ESTRELA}_b_0*.fits')
    arquivos_encontrados_i = sorted(glob.glob(padrao_busca))
    for arq in arquivos_encontrados_i:
        try:
            hdr = fits.getheader(arq)

            # O valor -999.0 serve como aviso caso a palavra-chave não exista
            valor = hdr.get('LAMINA', -999.0)

            valor = valor[1:]

            if valor != -999.0:
                valores_lamina.append(valor)             
        except Exception as e:
            print(f"Erro ao ler o ficheiro {os.path.basename(arq)}: {e}")

        valor_mais_alto = max(valores_lamina)
    #print(valor_mais_alto)

    

FileNotFoundError: [Errno 2] No such file or directory: '/home/likeableng/BeACoN/arquivos_de_reducao/25jul19/raw/48lib'

In [29]:
#Definindo a quantidade de lâminas e portanto a quantidade de imagens 
# por pasta de cada filtro de acordo com o nome da lâmina
if valor_mais_alto == "B":
    numero_de_arquivos_por_pasta = 8

else:
    numero_de_arquivos_por_pasta = 16
    

NameError: name 'valor_mais_alto' is not defined

In [ ]:
#Criando as pastas de cada observação da estrela

i=0 #Sei que da pra deixar aqui como um e fazer ser menor ou igual
    #, mas por algum motivo tava demorando mais de 30 seg (att - acho q tinha dado um loop infinitokkk)
while i < vezes_repetido:
    i=i+1 
    os.makedirs(f"{pasta_raw_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

    os.makedirs(f"{pasta_reduced_estrela}/Obs_{i}_{NOME_ESTRELA}", exist_ok=True)

In [ ]:
#Passando os arquivos de acordo com a numeração da observação

for nome_item in os.listdir(pasta_raw_estrela):
    caminho_origem_completo = os.path.join(pasta_raw_estrela, nome_item)

    if os.path.isfile(caminho_origem_completo):
    
        partes = nome_item.split('_')

        if len(partes) == 3:

            final_do_item = partes[2]
            valor_do_item = os.path.splitext(final_do_item)[0]  
            numero_do_item = int(valor_do_item)

            for i in range(vezes_repetido):
                limite_inferior = i * numero_de_arquivos_por_pasta
                limite_superior = (i + 1) * numero_de_arquivos_por_pasta
                
                if limite_inferior < numero_do_item <= limite_superior:
                    
                    pasta_raw_estrela_i = f"{pasta_raw_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
                    
                    if not os.path.exists(pasta_raw_estrela_i):
                        print(f"A pasta {pasta_raw_estrela_i} não existe")
                        break
                        
                    caminho_destino_completo = os.path.join(pasta_raw_estrela_i, nome_item)
                    shutil.move(caminho_origem_completo, caminho_destino_completo)
                    
                    break

# **Extração de dados**

In [ ]:
#Extrair os arquivos 2D dos fits 3D da pasta calib
arquivos = os.listdir(pasta_raw_calib)
tem_arquivos_2d = any(arquivo.count("_") >= 3 for arquivo in arquivos)

if tem_arquivos_2d:
    print("Pasta Calib já está com arquivos 2D")
else:
    descompactar_cubos_3d(pasta_raw_calib)

In [ ]:
#Extrair os arquivos 2D dos fits 3D da pasta da estrela

for i in range(vezes_repetido):

    pasta_raw_estrela_i = f"{pasta_raw_estrela}/Obs_{i+1}_{NOME_ESTRELA}"

    arquivos = os.listdir(pasta_raw_estrela_i)
    tem_arquivos_2d = any(arquivo.count("_") >= 3 for arquivo in arquivos)

    if tem_arquivos_2d:

        print(f"Pasta Obs_{i+1}_{NOME_ESTRELA} já está com arquivos 2D")
    else: 

        descompactar_cubos_3d(pasta_raw_estrela_i)

# **Processamento de Imagens**

In [ ]:
#Quando realmente criamos o MASTER BIAS
padrao_de_busca_em_bias = os.path.join(pasta_reduced_calib,"master_bias*.fits")
arquivos_master_bias_encontrados = glob.glob(padrao_de_busca_em_bias)
if arquivos_master_bias_encontrados:
    print("Master Bias já processados")
else:
    criar_master_bias(pasta_raw_calib, pasta_reduced_calib)

In [ ]:
#Quando realmente criamos o MASTER FLAT

padrao_de_busca_em_flat = os.path.join(pasta_reduced_calib,"master_flat*.fits")
arquivos_master_flat_encontrados = glob.glob(padrao_de_busca_em_flat)

if arquivos_master_flat_encontrados:
    print("Master Flat já processados")
else:
    criar_master_flat(pasta_raw_calib, pasta_reduced_calib)

In [ ]:
#Quando realmente calibramos a imagem

for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
    pasta_raw_estrela_i = f"{pasta_raw_estrela}/Obs_{i+1}_{NOME_ESTRELA}"

    arquivos = os.listdir(pasta_reduced_estrela_i)
    tem_arquivos_na_pasta = any(arquivo.endswith(".fits") for arquivo in arquivos)

    if tem_arquivos_na_pasta:

        print(f"Pasta Obs_{i+1}_{NOME_ESTRELA} já está com arquivos fits,"
              "certifique de que a calibração da imagem não foi feita, mesmo que parcialmente"
        )
    else: 

        reduzir_objetos_paralelo(pasta_raw_estrela_i, pasta_reduced_calib, pasta_reduced_estrela_i)

# **Encontrar o Par e Extração da Fotometria**

In [ ]:
# Parâmetros Fotométricos
raio_abertura = 6.0       # Raio da estrela (px)
r_in_sky = 10.0           # Raio interno do anel de céu (px)
r_out_sky = 15.0          # Raio externo do anel de céu (px)
janela_busca = 12         # Janela max de busca ao redor da coordenada âncora (px)

In [ ]:
# Area das aberturas para subtracao do ceu
area_aper = np.pi * (raio_abertura ** 2)
area_ann = np.pi * (r_out_sky ** 2 - r_in_sky ** 2)

In [ ]:
#Parâmetros das lâminas
mapa_lamina_graus = {
    'L0': 0.0,   'L1': 22.5,  'L2': 45.0,  'L3': 67.5,
    'L4': 90.0,  'L5': 112.5, 'L6': 135.0, 'L7': 157.5,
    'L8': 180.0, 'L9': 202.5, 'LA': 225.0, 'LB': 247.5,
    'LC': 270.0, 'LD': 292.5, 'LE': 315.0, 'LF': 337.5
}

In [ ]:
# Realizar a busca de pares e coleta de dados fotometricos em cada pasta

%matplotlib qt

for i in range(vezes_repetido):

    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
        
    todos_arquivos = glob.glob(os.path.join(pasta_reduced_estrela_i, 'red_*.fits'))

    if not todos_arquivos:
        raise ValueError("Nenhum arquivo 'red_*.fits' encontrado.")

    arquivos_por_filtro = defaultdict(list)
    for arq in todos_arquivos:
        nome = os.path.basename(arq)
        partes = nome.split('_')
        if len(partes) >= 5:
            filtro = partes[2]
            arquivos_por_filtro[filtro].append(arq)

    print("=== PIPELINE BLINDADO DE FOTOMETRIA BEACON ===")
    print(f"Filtros identificados: {list(arquivos_por_filtro.keys())}\n")

    # Loop por Filtro
    for filtro, arquivos in arquivos_por_filtro.items():
        arquivos = sorted(arquivos)
        total_arq = len(arquivos)
        
        print(f"\n{'='*60}")
        print(f"PROCESSANDO FILTRO: '{filtro.upper()}' ({total_arq} imagens)")
        print(f"{'='*60}")
        
        imagem_referencia = arquivos[0]
        data_ref = fits.getdata(imagem_referencia)
        data_ref = np.squeeze(data_ref)
        data_ref = np.nan_to_num(data_ref, nan=np.nanmedian(data_ref), posinf=0, neginf=0)

        # --- SELEÇÃO INTERATIVA POR CLIQUE DO MOUSE ---
        print(f"\n👉 [FILTRO {filtro.upper()}] Clique na imagem:")
        print("   1º Clique -> Estrela Principal (Feixe P)")
        print("   2º Clique -> Estrela Secundária (Feixe S)")

        fig, ax = plt.subplots(figsize=(8, 8))
        vmin = np.median(data_ref)
        vmax = np.percentile(data_ref, 99.5)
        ax.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax.set_title(f"CLIQUE INTERATIVO - Filtro {filtro.upper()}\n1º clique: Principal (P)  |  2º clique: Secundário (S)", fontsize=11, color='navy')
        plt.tight_layout()

        # Interrompe e aguarda 2 cliques na imagem
        pontos = plt.ginput(2, timeout=0)
        plt.close(fig)

        if len(pontos) < 2:
            print(f"⚠️ [AVISO] Seleção cancelada ou menos de 2 cliques detectados no filtro {filtro.upper()}. Pulando...")
            continue

        # Extrai coordenadas dos cliques
        (x_click_p, y_click_p), (x_click_s, y_click_s) = pontos

        # Refina a posição para o pico exato de brilho via centróide
        x_prin_ref, y_prin_ref = rastrear_centroide_ancorado(data_ref, x_click_p, y_click_p, janela=janela_busca)
        x_sec_ref, y_sec_ref = rastrear_centroide_ancorado(data_ref, x_click_s, y_click_s, janela=janela_busca)

        dx_rigido = x_sec_ref - x_prin_ref
        dy_rigido = y_sec_ref - y_prin_ref

        # --- Verificação humana ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        fig.suptitle(f"Confirmação: Filtro {filtro.upper()} | Ref: {os.path.basename(imagem_referencia)}")

        ax1.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax1.set_title(f"P: Principal ({x_prin_ref:.1f}, {y_prin_ref:.1f})")
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), raio_abertura, color='lime', fill=False, lw=2))
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), r_in_sky, color='yellow', fill=False, ls=':'))
        ax1.add_patch(Circle((x_prin_ref, y_prin_ref), r_out_sky, color='yellow', fill=False, ls=':'))
        ax1.set_xlim(x_prin_ref - 25, x_prin_ref + 25)
        ax1.set_ylim(y_prin_ref - 25, y_prin_ref + 25)

        ax2.imshow(data_ref, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        ax2.set_title(f"S: Secundário ({x_sec_ref:.1f}, {y_sec_ref:.1f})")
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), raio_abertura, color='red', fill=False, lw=2))
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), r_in_sky, color='yellow', fill=False, ls=':'))
        ax2.add_patch(Circle((x_sec_ref, y_sec_ref), r_out_sky, color='yellow', fill=False, ls=':'))
        ax2.set_xlim(x_sec_ref - 25, x_sec_ref + 25)
        ax2.set_ylim(y_sec_ref - 25, y_sec_ref + 25)

        plt.tight_layout()
        plt.show()

        confirma = input(f"Filtro {filtro.upper()}: As estrelas e anéis de céu estão corretos? (S/N): ").upper()
        if confirma != 'S' and confirma != '':
            continue

        # --- EXTRAÇÃO COM SUBTRAÇÃO DE CÉU E VÍNCULO RÍGIDO ---
        arquivo_saida = os.path.join(pasta_reduced_estrela_i, f'fotometria_filtro_{filtro}.csv')

        with open(arquivo_saida, 'w') as f_out:
            # Mantém o cabeçalho completo incluindo as colunas de pico
            f_out.write("Arquivo,Posicao,Angulo,X_Prin,Y_Prin,Fluxo_Prin,Pico_Prin,X_Sec,Y_Sec,Fluxo_Sec,Pico_Sec\n")
            
            fluxo_p_ref = None
            fluxo_s_ref = None
            ignorar_alertas = False

            for i, arq in enumerate(arquivos):
                nome = os.path.basename(arq)
                try:
                    hdr = fits.getheader(arq)
                    lamina_raw = str(hdr.get('LAMINA', 'L0')).strip().upper()
                    angulo_psi = mapa_lamina_graus.get(lamina_raw, 0.0)
                    posicao_int = int(angulo_psi / 22.5)
                except Exception as e:
                    posicao_int = 0
                    angulo_psi = 0.0
                
                dados = fits.getdata(arq)
                dados = np.squeeze(dados)
                dados = np.nan_to_num(dados, nan=np.nanmedian(dados), posinf=0, neginf=0)
                
                # 1. Rastreia o feixe principal
                x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                
                # 2. Vínculo rígido do feixe secundário
                x_sec = x_prin + dx_rigido
                y_sec = y_prin + dy_rigido
                
                # --- MEDIÇÃO DO PICO DE CONTAGENS (PARA DETECTAR SATURAÇÃO) ---
                h, w = dados.shape
                xi_p, yi_p = int(round(x_prin)), int(round(y_prin))
                pico_p = np.max(dados[max(0, yi_p-3):min(h, yi_p+4), max(0, xi_p-3):min(w, xi_p+4)])

                xi_s, yi_s = int(round(x_sec)), int(round(y_sec))
                pico_s = np.max(dados[max(0, yi_s-3):min(h, yi_s+4), max(0, xi_s-3):min(w, xi_s+4)])
                # -------------------------------------------------------------
                
                # 3. Fotometria com Anel de Céu
                aper_p = CircularAperture((x_prin, y_prin), r=raio_abertura)
                ann_p = CircularAnnulus((x_prin, y_prin), r_in=r_in_sky, r_out=r_out_sky)
                phot_p = aperture_photometry(dados, [aper_p, ann_p])
                
                aper_s = CircularAperture((x_sec, y_sec), r=raio_abertura)
                ann_s = CircularAnnulus((x_sec, y_sec), r_in=r_in_sky, r_out=r_out_sky)
                phot_s = aperture_photometry(dados, [aper_s, ann_s])
                
                bkg_p_pixel = phot_p['aperture_sum_1'][0] / area_ann
                fluxo_p = phot_p['aperture_sum_0'][0] - (bkg_p_pixel * area_aper)
                
                bkg_s_pixel = phot_s['aperture_sum_1'][0] / area_ann
                fluxo_s = phot_s['aperture_sum_0'][0] - (bkg_s_pixel * area_aper)
                
                if i == 0:
                    fluxo_p_ref = max(fluxo_p, 100.0)
                    fluxo_s_ref = max(fluxo_s, 100.0)

                # --- LÓGICA DO ALERTA INTELIGENTE ---
                desvio_posicao = np.sqrt((x_prin - x_prin_ref)**2 + (y_prin - y_prin_ref)**2)
                queda_fluxo_p = fluxo_p < (0.50 * fluxo_p_ref)
                queda_fluxo_s = fluxo_s < (0.50 * fluxo_s_ref)
                queda_critica = (fluxo_p < 0.20 * fluxo_p_ref) or (fluxo_s < 0.20 * fluxo_s_ref)
                deslocado_critico = desvio_posicao > 8.0

                ha_anomalia = (queda_critica or deslocado_critico)

                if ha_anomalia and not ignorar_alertas:
                    motivos = []
                    if deslocado_critico: motivos.append(f"Pulo de Posição ({desvio_posicao:.1f}px)")
                    if queda_fluxo_p or queda_fluxo_s: motivos.append("Queda de Sinal (>50% NUVEM/PERDA)")
                    motivo_str = " | ".join(motivos)
                    
                    print(f"\n⚠️  [ALERTA NA IMAGEM {i+1}/{total_arq}] {nome}")
                    print(f"   Motivo: {motivo_str}")
                    
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
                    fig.suptitle(f"⚠️ ALERTA: {motivo_str}\n{nome} [{i+1}/{total_arq}]", color='red', fontweight='bold')
                    
                    vmin, vmax = np.median(dados), np.percentile(dados, 99.5)
                    
                    ax1.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax1.add_patch(Circle((x_prin, y_prin), raio_abertura, color='lime', fill=False, lw=2))
                    ax1.add_patch(Circle((x_prin, y_prin), r_in_sky, color='yellow', fill=False, ls=':'))
                    ax1.add_patch(Circle((x_prin, y_prin), r_out_sky, color='yellow', fill=False, ls=':'))
                    ax1.set_xlim(x_prin - 25, x_prin + 25)
                    ax1.set_ylim(y_prin - 25, y_prin + 25)
                    ax1.set_title(f"Feixe P: Fluxo={fluxo_p:.0f} | Pico={pico_p:.0f}")
                    
                    ax2.imshow(dados, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
                    ax2.add_patch(Circle((x_sec, y_sec), raio_abertura, color='red', fill=False, lw=2))
                    ax2.add_patch(Circle((x_sec, y_sec), r_in_sky, color='yellow', fill=False, ls=':'))
                    ax2.add_patch(Circle((x_sec, y_sec), r_out_sky, color='yellow', fill=False, ls=':'))
                    ax2.set_xlim(x_sec - 25, x_sec + 25)
                    ax2.set_ylim(y_sec - 25, y_sec + 25)
                    ax2.set_title(f"Feixe S: Fluxo={fluxo_s:.0f} | Pico={pico_s:.0f}")
                    
                    plt.tight_layout()
                    plt.show()
                    
                    print("Opções de ação:\n [ENTER/S] Aceitar e Continuar\n [R] Recalibrar Posição Âncora\n [M] Mudar Referência de Fluxo\n [C] Continuar sem mais alertas")
                    resp = input("Escolha uma opção (ENTER/S/R/M/C): ").upper().strip()
                    if resp == 'R':
                        x_prin_ref = float(input(f"Novo X âncora (Atual: {x_prin:.1f}): "))
                        y_prin_ref = float(input(f"Novo Y âncora (Atual: {y_prin:.1f}): "))
                        x_prin, y_prin = rastrear_centroide_ancorado(dados, x_prin_ref, y_prin_ref, janela=janela_busca)
                        x_sec, y_sec = x_prin + dx_rigido, y_prin + dy_rigido
                    elif resp == 'M':
                        fluxo_p_ref, fluxo_s_ref = max(fluxo_p, 100.0), max(fluxo_s, 100.0)
                    elif resp == 'C':
                        ignorar_alertas = True

                if i % 50 == 0 or i == total_arq - 1:
                    print(f"[{i+1}/{total_arq}] {nome} | Pos {posicao_int} ({angulo_psi}°) -> F_Prin={fluxo_p:.0f} (Pico={pico_p:.0f}) | F_Sec={fluxo_s:.0f} (Pico={pico_s:.0f})")
                
                # Grava no CSV mantendo o layout das colunas
                f_out.write(f"{nome},{posicao_int},{angulo_psi:.1f},{x_prin:.3f},{y_prin:.3f},{fluxo_p:.3f},{pico_p:.1f},{x_sec:.3f},{y_sec:.3f},{fluxo_s:.3f},{pico_s:.1f}\n") 
                              
        print(f"✅ Filtro {filtro.upper()} finalizado e salvo em {arquivo_saida}!")

    print("\n=== RE-EXTRAÇÃO CONCLUÍDA COM SUCESSO! ===")

# **Dados Finais 1**

In [ ]:
# ====================================================================
# CONFIGURAÇÃO DE FILTRAGEM CIENTÍFICA POR CICLOS (ESTILO PyHDUST)
# ====================================================================
limites_saturacao = {
    'I': 9500.0,   # Saturação física no Filtro I
    'B': None,     # Preserva Filtro B
    'V': None,
    'R': None
}

limite_snr = 50.0  # SNR mínimo para rejeição de ruído

for i in range(vezes_repetido):
    
    pasta_reduced_estrela_i = f"{pasta_reduced_estrela}/Obs_{i+1}_{NOME_ESTRELA}"
    arquivos_csv = sorted(glob.glob(os.path.join(pasta_reduced_estrela_i, 'fotometria_filtro_*.csv')))

    if not arquivos_csv:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em {pasta_reduced_estrela_i}.")

    pol_solver = SLSDualBeamPolarimetry('halfwave', compute_k=True)
    resultados_finais = []

    for csv_file in arquivos_csv:
        filtro = os.path.basename(csv_file).split('_')[-1].replace('.csv', '').upper()
        pasta_origem = os.path.dirname(csv_file) 
        
        df = pd.read_csv(csv_file)
        
        # --------------------------------------------------------------------
        # 1. FILTRAGEM DE SEGURANÇA BASE
        # --------------------------------------------------------------------
        teto_pico = limites_saturacao.get(filtro, None)
        
        condicao = (df['Fluxo_Prin'] > 500) & (df['Fluxo_Sec'] > 500)
        
        if 'SNR_Prin' in df.columns and 'SNR_Sec' in df.columns:
            condicao = condicao & (df['SNR_Prin'] >= limite_snr) & (df['SNR_Sec'] >= limite_snr)
            
        if teto_pico is not None and 'Pico_Prin' in df.columns:
            condicao = condicao & (df['Pico_Prin'] < teto_pico) & (df['Pico_Sec'] < teto_pico)
            
        df_valido = df[condicao].copy()
        
        # --------------------------------------------------------------------
        # 2. CÁLCULO DE Zi E LIMPEZA ROBUSTA POR POSIÇÃO ANGULAR
        # --------------------------------------------------------------------
        df_valido['Zi'] = (df_valido['Fluxo_Sec'] - df_valido['Fluxo_Prin']) / (df_valido['Fluxo_Sec'] + df_valido['Fluxo_Prin'])
        df_valido = df_valido[np.abs(df_valido['Zi']) < 0.20].copy()

        if len(df_valido) == 0:
            print(f"[{filtro}] ⚠️ Nenhum dado válido restou após filtros primários.")
            continue

        # Identifica os ciclos observacionais
        df_valido['Ciclo'] = (df_valido['Angulo'] < df_valido['Angulo'].shift(1, fill_value=df_valido['Angulo'].iloc[0])).cumsum()

        # Filtragem por mediana em cada ângulo individual da lâmina (MAD_STD)
        from astropy.stats import sigma_clip
        indices_limpos = []
        for ang, group in df_valido.groupby('Angulo'):
            if len(group) > 2:
                res_ang = group['Zi'] - group['Zi'].median()
                clip_ang = sigma_clip(res_ang, sigma=2.5, stdfunc='mad_std')
                mask_ang = ~np.array(clip_ang.mask, dtype=bool) if not np.isscalar(clip_ang.mask) else np.ones(len(group), dtype=bool)
                indices_limpos.extend(group.index[mask_ang])
            else:
                indices_limpos.extend(group.index)
                
        df_limpo = df_valido.loc[indices_limpos].copy()

        # --------------------------------------------------------------------
        # 3. REJEIÇÃO DE CICLOS CORROMPIDOS
        # --------------------------------------------------------------------
        ciclos_validos = []
        for ciclo_id, group_c in df_limpo.groupby('Ciclo'):
            if group_c['Angulo'].nunique() >= 4:
                try:
                    psi_c = np.ascontiguousarray(group_c['Angulo'].values, dtype=np.float64)
                    fo_c = np.ascontiguousarray(group_c['Fluxo_Prin'].values, dtype=np.float64)
                    fe_c = np.ascontiguousarray(group_c['Fluxo_Sec'].values, dtype=np.float64)
                    
                    res_c = pol_solver.compute(psi_c, fo_c, fe_c)
                    if res_c.rms < 0.05 and (res_c.p.nominal * 100) < 3.0:
                        ciclos_validos.append(ciclo_id)
                except Exception:
                    pass
                    
        if ciclos_validos:
            df_final_fit = df_limpo[df_limpo['Ciclo'].isin(ciclos_validos)].copy()
        else:
            df_final_fit = df_limpo.copy()

        # --------------------------------------------------------------------
        # 4. CONVERSÃO SEGURA E AJUSTE POLARIMÉTRICO FINAL
        # --------------------------------------------------------------------
        psi = np.ascontiguousarray(df_final_fit['Angulo'].values, dtype=np.float64)
        fo = np.ascontiguousarray(df_final_fit['Fluxo_Prin'].values, dtype=np.float64)
        fe = np.ascontiguousarray(df_final_fit['Fluxo_Sec'].values, dtype=np.float64)

        # Remove eventuais valores não finitos
        mask_finitos = np.isfinite(psi) & np.isfinite(fo) & np.isfinite(fe)
        psi, fo, fe = psi[mask_finitos], fo[mask_finitos], fe[mask_finitos]

        if len(psi) < 4 or len(np.unique(psi)) < 4:
            print(f"[{filtro}] ⚠️ Insuficiência de ângulos válidos para o cálculo polarimétrico.")
            continue

        # Ajuste preliminar para estimar k
        res_preliminar = pol_solver.compute(psi, fo, fe)
        k_estimado = res_preliminar.k  # Salva o k preliminar
        
        residuos = res_preliminar.zi.nominal - res_preliminar.model(psi * u.deg).nominal
        resultado_clip = sigma_clip(residuos, sigma=2.5, stdfunc='mad_std')
        
        if np.isscalar(resultado_clip.mask):
            mascara_boa = np.ones(len(psi), dtype=bool)
        else:
            mascara_boa = ~np.array(resultado_clip.mask, dtype=bool)

        if np.sum(mascara_boa) < 4:
            mascara_boa = np.ones(len(psi), dtype=bool)
        
        # Tenta calcular recalculando k; se falhar por falta de ângulo no subset, fixa o k preliminar
        try:
            resultado_final = pol_solver.compute(psi[mascara_boa], fo[mascara_boa], fe[mascara_boa])
        except Exception:
            from astropop.polarimetry import SLSDualBeamPolarimetry
            pol_solver_fixed = SLSDualBeamPolarimetry('halfwave', compute_k=False, k=k_estimado)
            resultado_final = pol_solver_fixed.compute(psi[mascara_boa], fo[mascara_boa], fe[mascara_boa])
        
        n_total = len(df)
        n_usados = np.sum(mascara_boa)
        
        p_pct = resultado_final.p.nominal * 100
        p_err = resultado_final.p.std_dev * 100
        theta_deg = resultado_final.theta.nominal
        theta_err = resultado_final.theta.std_dev
        
        resultados_finais.append({
            'Filtro': filtro,
            'N_Usados': f"{n_usados}/{n_total}",
            'P (%)': f"{p_pct:.3f} ± {p_err:.3f}",
            'Theta (°)': f"{theta_deg:.2f} ± {theta_err:.2f}",
            'k': f"{resultado_final.k:.4f}",
            'RMS': f"{resultado_final.rms:.6f}"
        })
        
        # --------------------------------------------------------------------
        # 5. PLOTAGEM DA CURVA POLARIMÉTRICA
        # --------------------------------------------------------------------
        plt.figure(figsize=(8, 5))
        plt.errorbar(resultado_final.psi.nominal, resultado_final.zi.nominal, 
                    yerr=resultado_final.zi.std_dev, fmt='o', color='crimson', 
                    alpha=0.7, label=f'Medições Válidas (N={n_usados})')
        
        eixo_psi = np.linspace(0, 270, 600)
        modelo_zi = resultado_final.model(eixo_psi * u.deg).nominal
        plt.plot(eixo_psi, modelo_zi, '--', color='navy', lw=2, label='Modelo SLS ($Q\\cos 4\\psi + U\\sin 4\\psi$)')
        
        plt.title(f"Curva Polarimétrica Limpa - Filtro {filtro}\n$P = {p_pct:.2f}\\% \\pm {p_err:.2f}\\% \\quad \\theta = {theta_deg:.1f}^\\circ \\pm {theta_err:.1f}^\\circ$")
        plt.xlabel(r"Posição da Lâmina Retardadora $\psi$ (graus)")
        plt.ylabel(r"Diferença Relativa de Fluxo $Z_i$")
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='best')
        plt.tight_layout()
        
        nome_grafico = f"curva_polarimetrica_filtro_{filtro}.png"
        caminho_grafico = os.path.join(pasta_origem, nome_grafico)
        plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
        plt.show()

    # --------------------------------------------------------------------
    # 6. TABELA RESUMO FINAL
    # --------------------------------------------------------------------
    if resultados_finais:
        print("\n" + "="*70)
        print(f"  RESUMO DA POLARIMETRIA FINAL (OBS {i+1} - {NOME_ESTRELA.upper()})")
        print("="*70)
        df_resumo = pd.DataFrame(resultados_finais)
        tabela_texto = df_resumo.to_string(index=False)
        print(tabela_texto)

        caminho_out = os.path.join(pasta_reduced_estrela_i, f"resultados_polarimetria_obs{i+1}_{NOME_ESTRELA}.out")
        with open(caminho_out, 'w', encoding='utf-8') as f_out:
            f_out.write("="*70 + "\n")
            f_out.write(f"  RESUMO DA POLARIMETRIA FINAL - {NOME_ESTRELA.upper()} (OBS {i+1})\n")
            f_out.write("="*70 + "\n\n")
            f_out.write(tabela_texto + "\n")
        print(f"\n✅ Resultados numéricos salvos em: {caminho_out}\n")

In [ ]:
# Verifica o que realmente está no CSV do Filtro I
df_i = pd.read_csv(f"{pasta_reduced_estrela}/Obs_1_{NOME_ESTRELA}/fotometria_filtro_i.csv")

print("--- DIAGNÓSTICO DO CSV (FILTRO I) ---")
print(f"1. Colunas encontradas: {list(df_i.columns)}")
print(f"2. Total de imagens no CSV: {len(df_i)}")

if 'Pico_Prin' in df_i.columns:
    print(f"3. Quantas imagens têm Pico >= 4000 ADU: {(df_i['Pico_Prin'] >= 4000).sum()}")
    print(f"4. Quantas imagens têm Pico < 1000 ADU (céu/centroide perdido): {(df_i['Pico_Prin'] < 1000).sum()}")
else:
    print("⚠️ AVISO: As colunas de Pico e SNR NÃO existem neste CSV! Re-execute a Célula 39.")

In [ ]:
import matplotlib.pyplot as plt

df_i = pd.read_csv(f"{pasta_reduced_estrela}/Obs_1_{NOME_ESTRELA}/fotometria_filtro_i.csv")
df_i['Zi'] = (df_i['Fluxo_Sec'] - df_i['Fluxo_Prin']) / (df_i['Fluxo_Sec'] + df_i['Fluxo_Prin'])

plt.figure(figsize=(9, 4))
plt.plot(df_i['Angulo'], df_i['Zi'], 'o', alpha=0.5, color='purple', label='Dados Brutos Filtro I')
plt.axhline(0, color='black', linestyle='--')
plt.ylim(-0.25, 0.25)
plt.title("Distribuição do Zi no Filtro I vs Ângulo da Lâmina")
plt.xlabel("Ângulo (graus)")
plt.ylabel("Zi")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()